In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# 파일 경로 설정
file_path = '/content/drive/MyDrive/서울시 빅데이터 공모전/data/processed/subway_risk_result.csv'

In [ ]:
# 라이브러리 불러오기 + 데이터 로드
import pandas as pd
df = pd.read_csv(file_path, encoding='utf-8-sig')
df.head()

,역명_clean,호선명,time_group,Risk_Score,Risk,사고수,사고발생,위험유형,주요위험원인,혼잡기여도,유입기여도,구조기여도,환승기여도,추천대응전략
0,사당,2호선,퇴근시간,89.612410,0.896124,11,1,환승형,환승형,17.577361,16.471606,24.676271,41.274762,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"
1,사당,2호선,낮시간,89.576990,0.895770,12,1,환승형,환승형,17.273867,15.784539,25.046900,41.894694,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"
2,사당,4호선,낮시간,89.482670,0.894827,8,1,환승형,환승형,11.508750,25.662743,19.450347,43.378160,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"
3,종로3가,1호선,낮시간,89.417564,0.894176,4,1,구조형,구조형,12.316445,19.858253,33.977756,33.847546,"승강장 유도선 정비, 안전표지 보강, 시설 구조 점검"
4,신도림,2호선,낮시간,89.229870,0.892299,10,1,환승형,환승형,17.240165,11.438121,15.067968,56.253747,"환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화"


In [ ]:
# 위험등급 함수 정의
def risk_grade(score):
    if score >= 80:
        return '🔴 매우 위험', '1순위'
    elif score >= 60:
        return '🟠 위험', '2순위'
    elif score >= 40:
        return '🟡 주의', '3순위'
    else:
        return '🟢 보통', '4순위'

In [ ]:
# 위험유형 설명 정의
risk_type_desc = {
    '환승형': '환승 동선이 복잡하거나 환승 이용객 비중이 높아 승강장·통로 혼잡 및 보행 충돌 위험이 커지는 유형입니다.',
    '혼잡형': '승하차 인원과 시간대별 혼잡도가 높아 승강장 밀집 위험이 커지는 유형입니다.',
    '구조형': '승강장 이격거리 등 물리적 구조 위험 요인이 크게 작용하는 유형입니다.',
    '유입형': '역 주변 외부 유입 인구가 많아 출입구와 승강장으로 인파가 집중되는 유형입니다.'
}

In [ ]:
# MetroGuard-AI 조회 시스템 함수
def metroguard_query(station, line=None, time_group=None):
    result = df[df['역명_clean'].str.contains(station, na=False)]

    if line is not None:
        result = result[result['호선명'] == line]

    if time_group is not None:
        result = result[result['time_group'] == time_group]

    if result.empty:
        print("해당 조건의 결과가 없습니다.")
        return

    result = result.sort_values('Risk_Score', ascending=False)

    for _, row in result.iterrows():
        grade, priority = risk_grade(row['Risk_Score'])

        print("=" * 70)
        print("🚇 MetroGuard-AI 지하철 승강장 위험도 분석 결과")
        print("=" * 70)
        print(f"역명: {row['역명_clean']}")
        print(f"호선: {row['호선명']}")
        print(f"시간대: {row['time_group']}")
        print()
        print(f"Risk Score: {row['Risk_Score']:.1f}점")
        print(f"위험등급: {grade}")
        print(f"관리 우선순위: {priority}")
        print()
        print(f"위험유형: {row['위험유형']}")
        print("유형 설명:")
        print(risk_type_desc.get(row['위험유형'], '해당 위험유형에 대한 설명이 없습니다.'))
        print()
        main_cause = str(row['주요위험원인']).replace('형', ' 요인')
        print(f"주요 위험원인: {main_cause}")
        print()
        print("원인 기여도:")
        print(f"- 환승: {row['환승기여도']:.1f}%")
        print(f"- 구조: {row['구조기여도']:.1f}%")
        print(f"- 혼잡: {row['혼잡기여도']:.1f}%")
        print(f"- 유입: {row['유입기여도']:.1f}%")
        print()
        print("추천 대응전략:")
        print(row['추천대응전략'])
        print("=" * 70)
        print()

In [ ]:
# 실험해보기
metroguard_query('사당', '2호선', '퇴근시간')

🚇 MetroGuard-AI 지하철 승강장 위험도 분석 결과
역명: 사당
호선: 2호선
시간대: 퇴근시간

Risk Score: 89.6점
위험등급: 🔴 매우 위험
관리 우선순위: 1순위

위험유형: 환승형
유형 설명:
환승 동선이 복잡하거나 환승 이용객 비중이 높아 승강장·통로 혼잡 및 보행 충돌 위험이 커지는 유형입니다.

주요 위험원인: 환승 요인

원인 기여도:
- 환승: 41.3%
- 구조: 24.7%
- 혼잡: 17.6%
- 유입: 16.5%

추천 대응전략:
환승 동선 분리, 환승 통로 안전요원 배치, 환승 안내 표지 강화



In [ ]:
# 사용자 입력 기반 MetroGuard-AI 실행
# 역명, 호선, 시간대를 직접 입력해서 결과 조회

station_input = input("역명을 입력하세요: ")
line_input = input("호선을 입력하세요. 예: 2호선 / 전체 조회는 엔터: ")
time_input = input("시간대를 입력하세요. 예: 퇴근시간 / 전체 조회는 엔터: ")

# 엔터만 친 경우 None 처리
line_input = line_input if line_input.strip() != "" else None
time_input = time_input if time_input.strip() != "" else None

metroguard_query(station_input, line_input, time_input)

역명을 입력하세요: 미아사거리
호선을 입력하세요. 예: 2호선 / 전체 조회는 엔터: 4호선
시간대를 입력하세요. 예: 퇴근시간 / 전체 조회는 엔터: 퇴근시간
🚇 MetroGuard-AI 지하철 승강장 위험도 분석 결과
역명: 미아사거리
호선: 4호선
시간대: 퇴근시간

Risk Score: 5.9점
위험등급: 🟢 보통
관리 우선순위: 4순위

위험유형: 유입형
유형 설명:
역 주변 외부 유입 인구가 많아 출입구와 승강장으로 인파가 집중되는 유형입니다.

주요 위험원인: 유입 요인

원인 기여도:
- 환승: 0.0%
- 구조: 34.4%
- 혼잡: 14.4%
- 유입: 51.3%

추천 대응전략:
출입구 혼잡 관리, 피크시간 안전요원 배치, 유입 인구 분산 안내



In [ ]:
# 드롭다운 기반 MetroGuard-AI 조회 시스템
import ipywidgets as widgets
from IPython.display import display, clear_output

# 선택지 만들기
station_options = sorted(df['역명_clean'].dropna().unique())
line_options = ['전체'] + sorted(df['호선명'].dropna().unique())
time_options = ['전체'] + sorted(df['time_group'].dropna().unique())

# 드롭다운 UI
station_dropdown = widgets.Dropdown(
    options=station_options,
    description='역명:'
)

line_dropdown = widgets.Dropdown(
    options=line_options,
    description='호선:'
)

time_dropdown = widgets.Dropdown(
    options=time_options,
    description='시간대:'
)

button = widgets.Button(
    description='위험도 조회',
    button_style='danger'
)

output = widgets.Output()

def on_button_click(b):
    with output:
        clear_output()

        station = station_dropdown.value
        line = None if line_dropdown.value == '전체' else line_dropdown.value
        time_group = None if time_dropdown.value == '전체' else time_dropdown.value

        metroguard_query(station, line, time_group)

button.on_click(on_button_click)

display(station_dropdown, line_dropdown, time_dropdown, button, output)

Dropdown(description='역명:', options=('가락시장', '가산디지털단지', '가양', '강남', '강남구청', '강동', '강동구청', '강변', '강일', '개롱', '개…

Dropdown(description='호선:', options=('전체', '1호선', '2호선', '3호선', '4호선', '5호선', '6호선', '7호선', '8호선', '9호선'), val…

Dropdown(description='시간대:', options=('전체', '기타', '낮시간', '출근시간', '퇴근시간'), value='전체')

Button(button_style='danger', description='위험도 조회', style=ButtonStyle())

Output()

In [ ]:
# 최종: MetroGuard-AI 드롭다운 기반 위험도 조회 시스템

import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# 시스템 제목
display(Markdown("""
# 🚇 MetroGuard-AI 위험도 조회 시스템

역명, 호선, 시간대를 선택하면
해당 구간의 **Risk Score, 위험등급, 주요 원인, 대응전략**을 출력합니다.
"""))

# 역명 선택지
station_options = sorted(df['역명_clean'].dropna().unique())

# 드롭다운 UI
station_dropdown = widgets.Dropdown(
    options=station_options,
    description='역명:',
    layout=widgets.Layout(width='320px')
)

line_dropdown = widgets.Dropdown(
    options=[],
    description='호선:',
    layout=widgets.Layout(width='320px')
)

time_dropdown = widgets.Dropdown(
    options=[],
    description='시간대:',
    layout=widgets.Layout(width='320px')
)

button = widgets.Button(
    description='위험도 조회',
    button_style='danger',
    icon='search',
    layout=widgets.Layout(width='200px')
)

output = widgets.Output()


# 역명 선택 시 해당 역의 호선만 표시
def update_line_options(*args):
    station = station_dropdown.value

    lines = sorted(
        df[df['역명_clean'] == station]['호선명']
        .dropna()
        .unique()
    )

    line_dropdown.options = lines

    if len(lines) > 0:
        line_dropdown.value = lines[0]

    update_time_options()


# 역명 + 호선 선택 시 해당 조건의 시간대만 표시
def update_time_options(*args):
    station = station_dropdown.value
    line = line_dropdown.value

    times = sorted(
        df[
            (df['역명_clean'] == station) &
            (df['호선명'] == line)
        ]['time_group']
        .dropna()
        .unique()
    )

    time_dropdown.options = times

    if len(times) > 0:
        time_dropdown.value = times[0]


# 버튼 클릭 시 위험도 조회 실행
def on_button_click(b):
    with output:
        clear_output()

        station = station_dropdown.value
        line = line_dropdown.value
        time_group = time_dropdown.value

        metroguard_query(station, line, time_group)


# 드롭다운 값이 바뀔 때마다 선택지 자동 업데이트
station_dropdown.observe(update_line_options, names='value')
line_dropdown.observe(update_time_options, names='value')
button.on_click(on_button_click)

# 처음 실행 시 기본값 세팅
update_line_options()

# UI 출력
display(station_dropdown, line_dropdown, time_dropdown, button, output)


# 🚇 MetroGuard-AI 위험도 조회 시스템

역명, 호선, 시간대를 선택하면
해당 구간의 **Risk Score, 위험등급, 주요 원인, 대응전략**을 출력합니다.


Dropdown(description='역명:', layout=Layout(width='320px'), options=('가락시장', '가산디지털단지', '가양', '강남', '강남구청', '강동'…

Dropdown(description='호선:', layout=Layout(width='320px'), options=('3호선', '8호선'), value='3호선')

Dropdown(description='시간대:', layout=Layout(width='320px'), options=('기타', '낮시간', '출근시간', '퇴근시간'), value='기타')

Button(button_style='danger', description='위험도 조회', icon='search', layout=Layout(width='200px'), style=ButtonS…

Output()